# ICA — Microdados 2025
Padronização e consolidação dos microdados da Avaliação da Alfabetização (ICA) para o ano de 2025.

**Fonte:** [INEP — Resultados ICA](https://www.gov.br/inep/pt-br/areas-de-atuacao/avaliacao-e-exames-educacionais/avaliacao-da-alfabetizacao/resultados)

## Configuração

In [2]:
import os

import pandas as pd
from basedosdados import read_sql

/home/laribrito/BD/pipelines/.venv/lib/python3.10/site-packages/requests/__init__.py:102: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (5.2.0)/charset_normalizer (2.0.9) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({})/charset_normalizer ({}) doesn't match a supported "


In [3]:
ANO = "2025"
BASE_DIR = "models/br_inep_avaliacao_alfabetizacao"
INPUT_DIR = os.path.join(BASE_DIR, "input", f"ica_{ANO}")
OUTPUT_DIR = os.path.join(BASE_DIR, "output", f"ica_{ANO}")
BILLING_PROJECT = "basedosdados-dev"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Alunos

In [4]:
ALUNOS_INPUT = os.path.join(INPUT_DIR, "DADOS", "TS_ALUNO.csv")
ALUNOS_OUTPUT = os.path.join(OUTPUT_DIR, "alunos.csv")

ALUNOS_RENAME = {
    "NU_ANO_AVALIACAO": "ano",
    "CO_UF": "id_regiao",
    "SG_UF": "sigla_uf",
    "TP_SERIE": "serie",
    "TP_DEPENDENCIA": "rede",
    "CO_MUNICIPIO": "id_municipio",
    "IN_PRESENCA_LP": "presenca",
    "IN_PREENCHIMENTO_LP": "preenchimento_caderno",
    "CO_CADERNO_LP": "caderno",
    "VL_PESO_ALUNO_LP": "peso_aluno",
    "VL_PROFICIENCIA_LP": "proficiencia",
    "IN_ALFABETIZADO": "alfabetizado",
}

ID_COLS_ALUNOS = ["id_municipio", "id_escola", "rede"]

In [5]:
# Inspeciona colunas disponíveis
print(
    pd.read_csv(
        ALUNOS_INPUT, sep=";", encoding="latin1", nrows=0
    ).columns.tolist()
)

['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'ID_ALUNO', 'TP_SERIE', 'ID_ESCOLA', 'TP_DEPENDENCIA', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'IN_PRESENCA_LP', 'IN_PREENCHIMENTO_LP', 'CO_CADERNO_LP', 'CO_BLOCO_1', 'TX_RESPOSTA_BLOCO_1', 'TX_GABARITO_BLOCO_1', 'CO_BLOCO_2', 'TX_RESPOSTA_BLOCO_2', 'TX_GABARITO_BLOCO_2', 'CO_BLOCO_3', 'TX_RESPOSTA_BLOCO_3', 'TX_GABARITO_BLOCO_3', 'CO_BLOCO_4', 'TX_RESPOSTA_BLOCO_4', 'TX_GABARITO_BLOCO_4', 'VL_PESO_ALUNO_LP', 'VL_PROFICIENCIA_LP', 'IN_ALFABETIZADO']


In [ ]:
# Lê, renomeia e padroniza
df_alunos = pd.read_csv(ALUNOS_INPUT, sep=";", encoding="latin1")
df_alunos = df_alunos.rename(columns=ALUNOS_RENAME)
df_alunos.columns = df_alunos.columns.str.lower()

# Converte colunas de ID para string
for col in ID_COLS_ALUNOS:
    if col in df_alunos.columns:
        df_alunos[col] = df_alunos[col].astype("Int64").astype(str)

In [ ]:
# Filtra apenas colunas presentes no schema de destino (basedosdados-dev)
df_schema = read_sql(
    query="SELECT * FROM `basedosdados-dev.br_inep_avaliacao_alfabetizacao.alunos` LIMIT 0",
    billing_project_id=BILLING_PROJECT,
)

cols_comuns = [c for c in df_schema.columns if c in df_alunos.columns]
df_alunos = df_alunos[cols_comuns]

In [ ]:
# Concatena com dados históricos (2023/2024) e salva
df_alunos_historico = read_sql(
    query="SELECT * FROM `basedosdados-dev.br_inep_avaliacao_alfabetizacao.alunos`",
    billing_project_id=BILLING_PROJECT,
)

df_alunos_final = pd.concat(
    [df_alunos, df_alunos_historico], ignore_index=True
)
df_alunos_final.to_csv(ALUNOS_OUTPUT, index=False)

print(df_alunos_final.shape)
df_alunos_final.head()

## 2. Município

In [41]:
MUNICIPIO_INPUT = os.path.join(INPUT_DIR, "DADOS", "TS_MUNICIPIO.csv")
MUNICIPIO_OUTPUT = os.path.join(OUTPUT_DIR, "municipio.csv")

MUNICIPIO_RENAME = {
    "NU_ANO_AVALIACAO": "ano",
    "SG_UF": "sigla_uf",
    "TP_SERIE": "serie",
    "ID_TIPO_REDE": "rede",
    "CO_MUNICIPIO": "id_municipio",
    "PC_ALUNO_ALFABETIZADO": "taxa_alfabetizacao",
    "VL_MEDIA_LP": "media_portugues",
    **{
        f"PC_ALUNO_NIVEL_{i}_LP": f"proporcao_aluno_nivel_{i}"
        for i in range(9)
    },
}

In [37]:
# Inspeciona colunas disponíveis
print(
    pd.read_csv(
        MUNICIPIO_INPUT, sep=";", encoding="latin1", nrows=0
    ).columns.tolist()
)

['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'TP_SERIE', 'ID_TIPO_REDE', 'PC_ALUNO_ALFABETIZADO', 'VL_MEDIA_LP', 'PC_ALUNO_NIVEL_0_LP', 'PC_ALUNO_NIVEL_1_LP', 'PC_ALUNO_NIVEL_2_LP', 'PC_ALUNO_NIVEL_3_LP', 'PC_ALUNO_NIVEL_4_LP', 'PC_ALUNO_NIVEL_5_LP', 'PC_ALUNO_NIVEL_6_LP', 'PC_ALUNO_NIVEL_7_LP', 'PC_ALUNO_NIVEL_8_LP']


In [42]:
# Lê, renomeia e seleciona colunas
df_municipio = pd.read_csv(MUNICIPIO_INPUT, sep=";", encoding="latin1")
df_municipio = df_municipio.rename(columns=MUNICIPIO_RENAME)
df_municipio.columns = df_municipio.columns.str.lower()

# Filtra apenas colunas presentes no schema de destino (basedosdados-dev)
df_schema = read_sql(
    query="SELECT * FROM `basedosdados-dev.br_inep_avaliacao_alfabetizacao.municipio` LIMIT 0",
    billing_project_id=BILLING_PROJECT,
)

cols_comuns = [c for c in df_schema.columns if c in df_municipio.columns]
df_municipio = df_municipio[cols_comuns]

Downloading: |          |


In [39]:
df_municipio

,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2025,1100015,2,3,78.15,763.2938,0.71,2.92,5.59,7.04,10.22,32.52,25.25,11.54,4.21
1,2025,1100015,2,5,78.15,763.2938,0.71,2.92,5.59,7.04,10.22,32.52,25.25,11.54,4.21
2,2025,1100023,2,5,79.50,765.1730,1.61,2.40,3.98,6.47,10.20,30.22,28.52,13.14,3.45
3,2025,1100023,2,3,79.50,765.1730,1.61,2.40,3.98,6.47,10.20,30.22,28.52,13.14,3.45
4,2025,1100031,2,5,86.89,768.9794,0.00,0.00,4.92,3.28,8.20,42.63,32.79,4.91,3.28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12411,2025,5222203,2,5,86.22,773.0700,0.00,0.00,0.00,11.99,1.79,26.86,47.38,11.98,0.00
12412,2025,5222302,2,3,87.57,775.3300,0.00,0.00,1.34,5.41,8.29,31.56,34.43,16.29,2.68
12413,2025,5222302,2,5,87.57,775.3300,0.00,0.00,1.34,5.41,8.29,31.56,34.43,16.29,2.68
12414,2025,5300108,2,2,64.84,747.3454,2.81,4.07,7.25,8.64,20.04,33.62,18.57,4.08,0.91


In [43]:
# Concatena com dados históricos (staging) e salva
df_municipio_historico = read_sql(
    query="SELECT * FROM `basedosdados-dev.br_inep_avaliacao_alfabetizacao.municipio`",
    billing_project_id=BILLING_PROJECT,
)

df_municipio_final = pd.concat(
    [df_municipio, df_municipio_historico], ignore_index=True
)
df_municipio_final.to_csv(MUNICIPIO_OUTPUT, index=False)

print(df_municipio_final.shape)
df_municipio_final.head()

Downloading: 100%|██████████|
(36411, 15)


,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2025,1100015,2,3,78.15,763.2938,0.71,2.92,5.59,7.04,10.22,32.52,25.25,11.54,4.21
1,2025,1100015,2,5,78.15,763.2938,0.71,2.92,5.59,7.04,10.22,32.52,25.25,11.54,4.21
2,2025,1100023,2,5,79.50,765.1730,1.61,2.40,3.98,6.47,10.20,30.22,28.52,13.14,3.45
3,2025,1100023,2,3,79.50,765.1730,1.61,2.40,3.98,6.47,10.20,30.22,28.52,13.14,3.45
4,2025,1100031,2,5,86.89,768.9794,0.00,0.00,4.92,3.28,8.20,42.63,32.79,4.91,3.28


## 3. UF

In [26]:
import os

UF_INPUT = os.path.join(INPUT_DIR, "DADOS", "TS_ESTADO.csv")
UF_OUTPUT = os.path.join(OUTPUT_DIR, "uf.csv")

UF_RENAME = {
    "NU_ANO_AVALIACAO": "ano",
    "SG_UF": "sigla_uf",
    "TP_SERIE": "serie",
    "ID_TIPO_REDE": "rede",
    "PC_ALUNO_ALFABETIZADO": "taxa_alfabetizacao",
    "VL_MEDIA_LP": "media_portugues",
    **{
        f"PC_ALUNO_NIVEL_{i}_LP": f"proporcao_aluno_nivel_{i}"
        for i in range(9)
    },
}

In [27]:
# Inspeciona colunas disponíveis
print(
    pd.read_csv(UF_INPUT, sep=";", encoding="latin1", nrows=0).columns.tolist()
)

['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'TP_SERIE', 'ID_TIPO_REDE', 'PC_ALUNO_ALFABETIZADO', 'VL_MEDIA_LP', 'PC_ALUNO_NIVEL_0_LP', 'PC_ALUNO_NIVEL_1_LP', 'PC_ALUNO_NIVEL_2_LP', 'PC_ALUNO_NIVEL_3_LP', 'PC_ALUNO_NIVEL_4_LP', 'PC_ALUNO_NIVEL_5_LP', 'PC_ALUNO_NIVEL_6_LP', 'PC_ALUNO_NIVEL_7_LP', 'PC_ALUNO_NIVEL_8_LP']


In [28]:
# Lê, renomeia e seleciona colunas
df_uf = pd.read_csv(UF_INPUT, sep=";", encoding="latin1")
df_uf = df_uf.rename(columns=UF_RENAME)
df_uf.columns = df_uf.columns.str.lower()

df_schema = read_sql(
    query="SELECT * FROM `basedosdados-dev.br_inep_avaliacao_alfabetizacao.uf` LIMIT 0",
    billing_project_id=BILLING_PROJECT,
)
cols_comuns = [c for c in df_schema.columns if c in df_uf.columns]
df_uf = df_uf[cols_comuns]

Downloading: |          |


In [29]:
df_uf

,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2025,RO,2,2,77.66,763.9429,1.04,1.63,4.97,7.54,13.13,30.33,26.43,11.12,3.81
1,2025,RO,2,5,75.30,761.7828,1.73,2.49,5.07,7.93,12.37,29.96,25.65,11.10,3.70
2,2025,RO,2,3,75.16,761.6562,1.77,2.54,5.08,7.96,12.32,29.94,25.60,11.10,3.69
3,2025,AC,2,5,67.99,752.7504,3.08,4.18,8.73,4.37,19.49,27.27,20.94,9.39,2.56
4,2025,AC,2,2,71.12,756.9630,2.54,3.77,7.75,4.19,17.84,26.70,23.21,10.55,3.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,2025,GO,2,5,80.33,770.4500,1.17,1.81,3.81,7.30,8.69,24.62,31.86,15.46,5.28
76,2025,GO,2,3,80.33,770.4500,1.17,1.81,3.81,7.30,8.69,24.62,31.86,15.46,5.29
77,2025,GO,2,2,84.09,776.4500,0.00,0.00,2.27,6.82,11.36,22.73,34.09,22.73,0.00
78,2025,DF,2,5,64.84,747.3454,2.81,4.07,7.25,8.64,20.04,33.62,18.57,4.08,0.91


In [30]:
# Concatena com dados históricos (staging) e salva
df_uf_historico = read_sql(
    query="SELECT * FROM `basedosdados-dev.br_inep_avaliacao_alfabetizacao.uf`",
    billing_project_id=BILLING_PROJECT,
)

df_uf_final = pd.concat([df_uf, df_uf_historico], ignore_index=True)
df_uf_final.to_csv(UF_OUTPUT, index=False)

print(df_uf_final.shape)
df_uf_final.head()

Downloading: 100%|██████████|
(225, 15)


,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2025,RO,2,2,77.66,763.9429,1.04,1.63,4.97,7.54,13.13,30.33,26.43,11.12,3.81
1,2025,RO,2,5,75.30,761.7828,1.73,2.49,5.07,7.93,12.37,29.96,25.65,11.10,3.70
2,2025,RO,2,3,75.16,761.6562,1.77,2.54,5.08,7.96,12.32,29.94,25.60,11.10,3.69
3,2025,AC,2,5,67.99,752.7504,3.08,4.18,8.73,4.37,19.49,27.27,20.94,9.39,2.56
4,2025,AC,2,2,71.12,756.9630,2.54,3.77,7.75,4.19,17.84,26.70,23.21,10.55,3.45
